# Modelo de percepción de inseguridad — Centro Histórico de Quito, 2024-2026

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `261030_CHQ_unificado_LC_CA_PF_limpia.xlsx` (y, para la verificación final, `261030_modelo_percepcion_CHQ_v2.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels", "openpyxl"], check=False)

DATA_FILES = ["261030_CHQ_unificado_LC_CA_PF_limpia.xlsx"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

## 1. Construcción de la base armonizada
Desde la base unificada del OMSCGR (hoja *Base LCCA*) según `libro_de_codigos.csv`:
Sí/No → 1/0; "Ns/Nc" y vacíos → perdido; PSC.1 Inseguro/Muy inseguro = 1.
Cada pregunta se identifica por su código (PSC.1, VI.1, SSM.3/7/8, CC.1-9, CC.13, DD. 1).

In [ ]:
# Carga y armonización
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
warnings.filterwarnings('ignore')

raw = pd.read_excel(DATA_PATH, sheet_name='Base LCCA')

def col(prefijo, exacto=False):
    m = [c for c in raw.columns if (str(c).strip() == prefijo if exacto else str(c).strip().startswith(prefijo))]
    if len(m) != 1:
        raise ValueError(f'Pregunta {prefijo!r}: {len(m)} columnas coinciden')
    return m[0]

def sino(s):
    return s.astype(str).str.strip().str.lower().map({'si': 1, 'sí': 1, 'no': 0}).astype(float)

PREG = {'victima': 'VI.1 ', 'vio_ffaa': 'SSM.8 ', 'vio_policia': 'SSM.7 ', 'vio_cacmq': 'SSM.3 ',
        'cc_peleas': 'CC.1 ', 'cc_situacion_calle': 'CC.2 ', 'cc_alcohol': 'CC.3 ', 'cc_drogas': 'CC.4 ',
        'cc_danio_propiedad': 'CC.5 ', 'cc_fauna_urbana': 'CC.6 ', 'cc_basura': 'CC.7 ', 'cc_ruido': 'CC.8 ',
        'cc_aglomeracion': 'CC.9 ', 'ferias': 'CC.13 '}
data = pd.DataFrame()
data['anio'] = pd.to_numeric(raw[col('Año', True)], errors='coerce')
fuera_ciclo = raw[col('Mes', True)].astype(str).str.lower().str.contains('febrero')
data['tipo_actor'] = raw[col('Tipo', True)].astype(str).str.strip().str.lower().map(
    {'población flotante': 'flotante', 'local comercial': 'locales', 'comercio autónomo regularizado': 'autonomo'})
data['inseguro'] = raw[col('PSC.1 ')].astype(str).str.strip().str.lower().map(
    {'inseguro': 1, 'muy inseguro': 1, 'seguro': 0, 'muy seguro': 0}).astype(float)
for v, p in PREG.items():
    data[v] = sino(raw[col(p)])
data['hombre'] = raw[col('DD. 1')].astype(str).str.strip().str.lower().map({'hombre': 1, 'mujer': 0}).astype(float)
data = data[~fuera_ciclo & data['anio'].isin([2024, 2025, 2026])].copy()
data['anio'] = data['anio'].astype(int)
assert data['tipo_actor'].notna().all()
print('Filas brutas:', len(raw), '| en la base armonizada:', len(data))
print(data.groupby(['anio', 'tipo_actor']).size().unstack())

In [ ]:
# Etiquetas y muestra analítica (2024 y 2026, casos completos)
ACTORES = {'flotante': 'Población flotante', 'locales': 'Locales comerciales', 'autonomo': 'Comercio autónomo'}
CC = ['cc_peleas', 'cc_situacion_calle', 'cc_alcohol', 'cc_drogas', 'cc_danio_propiedad',
      'cc_fauna_urbana', 'cc_basura', 'cc_ruido', 'cc_aglomeracion']
INST = ['vio_cacmq', 'vio_policia', 'vio_ffaa']
ETIQ = {'anio2026': 'Año 2026 (vs. 2024, a igual composición)', 'hombre': 'Ser hombre (vs. mujer)',
        'victima': 'Haber sido víctima de un delito', 'ferias': 'Haber presenciado ferias gastronómicas/culturales',
        'cc_peleas': 'Peleas o riñas en el espacio público', 'cc_situacion_calle': 'Personas en situación de calle',
        'cc_alcohol': 'Consumo de alcohol en el espacio público', 'cc_drogas': 'Consumo de drogas en el espacio público',
        'cc_danio_propiedad': 'Daño a la propiedad pública/privada', 'cc_fauna_urbana': 'Fauna urbana (animales sin hogar)',
        'cc_basura': 'Basura en los espacios públicos', 'cc_ruido': 'Ruidos molestos', 'cc_aglomeracion': 'Aglomeración de personas',
        'actor_flotante': 'Ser población flotante (vs. comercio autónomo)', 'actor_locales': 'Ser local comercial (vs. comercio autónomo)',
        'vio_cacmq': 'Vio trabajar al CACMQ', 'vio_policia': 'Vio trabajar a la Policía Nacional', 'vio_ffaa': 'Vio trabajar a las Fuerzas Armadas'}

d2 = data[data['anio'].isin([2024, 2026])]
df = d2.dropna().copy()
df['anio2026'] = (df['anio'] == 2026).astype(int)
df['actor_flotante'] = (df['tipo_actor'] == 'flotante').astype(int)
df['actor_locales'] = (df['tipo_actor'] == 'locales').astype(int)
COV = ['anio2026', 'hombre', 'victima', 'ferias'] + CC + INST + ['actor_flotante', 'actor_locales']
DCOV = [c for c in COV if c != 'anio2026']
print('N analítico =', len(df), df.groupby('anio').size().to_dict(), '| exclusiones =', len(d2) - len(df))

In [ ]:
# Trayectorias por actor (las tres olas, total de encuestas)
c1 = data.groupby(['tipo_actor', 'anio'])[['inseguro', 'victima']].mean() * 100
serie = data.groupby('anio')[['inseguro', 'victima']].mean() * 100
medias = df.groupby('anio')[['inseguro', 'victima']].mean() * 100
print(c1.round(1).unstack()); print(serie.round(1))

In [ ]:
# Modelo logit agrupado y efectos marginales promedio
def logit(d, cov):
    X = sm.add_constant(d[cov].astype(float), has_constant='add')
    return sm.Logit(d['inseguro'].astype(float), X).fit(disp=0, cov_type='HC1')

m = logit(df, COV)
ame = m.get_margeff(at='overall').summary_frame().rename(columns={'dy/dx': 'AME', 'Std. Err.': 'EE', 'Pr(>|z|)': 'p'})[['AME', 'EE', 'p']]
ame = ame.sort_values('AME', key=np.abs, ascending=False)
print(f'pseudo-R² = {m.prsquared:.3f}'); print((ame * [100, 100, 1]).round(3))

In [ ]:
# Descomposición Oaxaca-Blinder/Fairlie (referencias 2024 y 2026) y detalle de Yun
def descomponer(d24, d26):
    X24 = sm.add_constant(d24[DCOV].astype(float), has_constant='add')
    X26 = sm.add_constant(d26[DCOV].astype(float), has_constant='add')
    m24 = sm.Logit(d24['inseguro'].astype(float), X24).fit(disp=0, cov_type='HC1')
    m26 = sm.Logit(d26['inseguro'].astype(float), X26).fit(disp=0, cov_type='HC1')
    P = lambda mod, X: mod.predict(X).mean()
    p24, p26 = P(m24, X24), P(m26, X26)
    r = {'p24': p24, 'p26': p26, 'brecha': p24 - p26,
         'expl_24': p24 - P(m24, X26), 'noexpl_24': P(m24, X26) - p26,
         'expl_26': P(m26, X24) - p26, 'noexpl_26': p24 - P(m26, X24), 'm24': m24, 'm26': m26}
    xbar = pd.concat([X24, X26]).mean()
    for ref, mod in (('24', m24), ('26', m26)):
        pb = 1 / (1 + np.exp(-(xbar * mod.params).sum()))
        r[f'detalle_{ref}'] = {k: -mod.params[k] * (X26[k].mean() - X24[k].mean()) * pb * (1 - pb) * 100 for k in DCOV}
    return r

d24, d26 = df[df['anio'] == 2024], df[df['anio'] == 2026]
D = descomponer(d24, d26)
print(f"Brecha = {D['brecha']*100:.2f} p.p. | explicado ref24 = {D['expl_24']/D['brecha']*100:.1f}% | ref26 = {D['expl_26']/D['brecha']*100:.1f}%")

In [ ]:
# Bootstrap de la descomposición (B = 300, semilla fija)
rng = np.random.default_rng(20260924)
B, boot = 300, []
for b in range(B):
    s24 = d24.sample(len(d24), replace=True, random_state=rng.integers(1e9))
    s26 = d26.sample(len(d26), replace=True, random_state=rng.integers(1e9))
    try:
        r = descomponer(s24, s26)
        boot.append([r['brecha']*100, r['expl_24']/r['brecha']*100, r['expl_26']/r['brecha']*100,
                     r['detalle_24']['vio_ffaa'], r['detalle_26']['vio_ffaa'],
                     r['detalle_24']['vio_policia'], r['detalle_26']['vio_policia'],
                     r['detalle_24']['cc_situacion_calle'], r['detalle_26']['cc_situacion_calle']])
    except Exception:
        pass
BCOL = ['brecha', 'pct_expl_24', 'pct_expl_26', 'ffaa_24', 'ffaa_26', 'policia_24', 'policia_26', 'calle_24', 'calle_26']
boot = pd.DataFrame(boot, columns=BCOL)
q = boot.quantile([0.025, 0.5, 0.975])
print('Réplicas exitosas:', len(boot)); print(q.T.round(2))

In [ ]:
# APP_DATA: todo lo que dibuja el HTML
prev = df.groupby('anio')[DCOV].mean() * 100
g = D['brecha']
det_rank = {ref: pd.Series(D[f'detalle_{ref}']).rank(ascending=False).astype(int) for ref in ('24', '26')}
A = {
    'meta': {'base': 'Base LCCA (base unificada OMSCGR, 261030)', 'olas': [2024, 2025, 2026]},
    'conciliacion': {'filas_brutas': int(len(raw)), 'filas_base': int(len(data)),
                     'encuestas_por_ola': {str(k): int(v) for k, v in data.groupby('anio').size().items()},
                     'encuestas_2024_2026': int(len(d2)), 'n_analitico': int(len(df)),
                     'n_analitico_por_ola': {str(k): int(v) for k, v in df.groupby('anio').size().items()},
                     'exclusiones': int(len(d2) - len(df))},
    'trayectorias': [{'actor': lab, 'clave': k,
                      'inseguro': [float(c1.loc[(k, y), 'inseguro']) for y in (2024, 2025, 2026)],
                      'victima': [float(c1.loc[(k, y), 'victima']) for y in (2024, 2025, 2026)]} for k, lab in ACTORES.items()]
                    + [{'actor': 'Total Centro Histórico', 'clave': 'total',
                        'inseguro': [float(serie.loc[y, 'inseguro']) for y in (2024, 2025, 2026)],
                        'victima': [float(serie.loc[y, 'victima']) for y in (2024, 2025, 2026)]}],
    'muestra_analitica': {'inseguro_2024': float(medias.loc[2024, 'inseguro']), 'inseguro_2026': float(medias.loc[2026, 'inseguro']),
                          'victima_2024': float(medias.loc[2024, 'victima']), 'victima_2026': float(medias.loc[2026, 'victima'])},
    'modelo': {'n': int(m.nobs), 'pseudo_r2': float(m.prsquared)},
    'ame': [{'var': v, 'label': ETIQ[v], 'ame_pp': float(ame.loc[v, 'AME'] * 100), 'ee_pp': float(ame.loc[v, 'EE'] * 100),
             'p': float(ame.loc[v, 'p'])} for v in ame.index],
    'descomposicion': {'p24': float(D['p24'] * 100), 'p26': float(D['p26'] * 100), 'brecha_pp': float(g * 100),
                       'ref24': {'explicado_pp': float(D['expl_24'] * 100), 'noexpl_pp': float(D['noexpl_24'] * 100),
                                 'explicado_pct': float(D['expl_24'] / g * 100), 'noexpl_pct': float(D['noexpl_24'] / g * 100)},
                       'ref26': {'explicado_pp': float(D['expl_26'] * 100), 'noexpl_pp': float(D['noexpl_26'] * 100),
                                 'explicado_pct': float(D['expl_26'] / g * 100), 'noexpl_pct': float(D['noexpl_26'] / g * 100)}},
    'detalle': {ref: [{'var': k, 'label': ETIQ[k], 'val': float(D[f'detalle_{ref}'][k]), 'orden': int(det_rank[ref][k]),
                       'prev_2024': float(prev.loc[2024, k]), 'prev_2026': float(prev.loc[2026, k])}
                      for k in sorted(DCOV, key=lambda k: D[f'detalle_{ref}'][k])] for ref in ('24', '26')},
    'instituciones_suma': {ref: float(sum(D[f'detalle_{ref}'][k] for k in INST)) for ref in ('24', '26')},
    'ffaa_coef': {'b2024': float(D['m24'].params['vio_ffaa']), 'p2024': float(D['m24'].pvalues['vio_ffaa']),
                  'b2026': float(D['m26'].params['vio_ffaa']), 'p2026': float(D['m26'].pvalues['vio_ffaa'])},
    'bootstrap': {'B': B, 'exitosas': int(len(boot)),
                  **{c: {'lo': float(q.loc[0.025, c]), 'med': float(q.loc[0.5, c]), 'hi': float(q.loc[0.975, c])} for c in BCOL}},
}
print('APP_DATA listo:', list(A.keys()))

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261030_modelo_percepcion_CHQ_v2.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")